In [ ]:
"""
Build a target-aware unified 4-class citation-function dataset from:

1) ACT2_dataset.tsv
2) full-v20210918.json (MultiCite)
3) adsabs/FOCAL (downloaded with Hugging Face datasets)
4) journal_citation_function_real.csv
5) synthetic_citation_funtion_40923.csv

Final CSV headers:
    source_dataset
    citation
    original_label
    follow_up

Target-aware convention:
    [TARGET_CITATION] = the specific citation/target mention whose function is labeled
    [OTHER_CITATION]  = every other detectable citation marker in the same context

Unified follow_up labels:
    0 = background_related_work
    1 = comparison
    2 = using_the_work
    3 = extending_the_work

Important source-specific rules
-------------------------------
ACT2
    - Uses #CITATION_TAG as the annotated target marker.
    - If a context contains more than one #CITATION_TAG, one target-aware
      instance is created per occurrence: one becomes [TARGET_CITATION] and
      the remaining target placeholders become [OTHER_CITATION].

MultiCite
    - Uses the original yellow HTML <span> annotations.
    - The script prefers an explicitly bibliographic highlighted span
      (author-year or numeric citation). If none is available, it falls back
      to another highlighted target mention/coreference in the annotated
      citation-bearing context.
    - Exactly one highlighted occurrence becomes [TARGET_CITATION].
      Other highlighted mentions are restored as normal text; any other
      detectable formal citation markers are then converted to [OTHER_CITATION].
    - If no highlighted target span is available in the selected gold context,
      the record is skipped rather than guessed.

FOCAL
    - Uses "Citation Start End" character offsets relative to "Paragraph".
    - Only the sentence containing the annotated target citation is retained.
    - The exact annotated target span becomes [TARGET_CITATION].
    - Other detectable citation markers in that sentence become [OTHER_CITATION].

JOURNAL_REAL
    - Loads journal_citation_function_real.csv.
    - Already target-aware; every row must contain exactly one [TARGET_CITATION].
    - All rows are required. Invalid rows raise an error.

SYNTHETIC
    - Loads synthetic_citation_funtion_40923.csv.
    - Already target-aware; every row must contain exactly one [TARGET_CITATION].
    - Redundant wrappers such as ([TARGET_CITATION]) are normalized.
    - All rows are required. Invalid rows raise an error.

Strict label policy
-------------------
MultiCite and FOCAL may contain multiple labels.
    - If all mapped labels resolve to one unified class, keep the record.
    - If mapped labels resolve to different unified classes, exclude it.
    - If a target label co-occurs with a non-target label
      (Motivation/Future Work/Unsure/etc.), exclude it.

Outputs
-------
1) unified_citation_function_4class.csv
2) unified_citation_function_4class_target_aware_audit.csv

This is a pure Python script. It does not use WandB.
"""

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# Set WANDB_API_KEY as an environment variable (never commit an API key).

In [ ]:
!pip install wandb -qU  # Εγκατάσταση της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login()

In [ ]:
from __future__ import annotations

import csv
import html
import json
import re
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

In [ ]:
# ============================================================
# 1. PATHS
# ============================================================

BASE_DIR = Path("/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER")

ACT2_PATH = BASE_DIR / "ACT2_dataset.tsv"
MULTICITE_PATH = BASE_DIR / "full-v20210918.json"
JOURNAL_PATH = BASE_DIR / "journal_citation_function_real.csv"
SYNTHETIC_PATH = BASE_DIR / "synthetic_citation_function_40923.csv"

OUTPUT_CSV = BASE_DIR / "unified_citation_function_4class.csv"
AUDIT_CSV = BASE_DIR / "unified_citation_function_4class_target_aware_audit.csv"

In [ ]:
# ============================================================
# 2. TARGET-AWARE TOKENS + LABEL SPACE
# ============================================================

TARGET_TOKEN = "[TARGET_CITATION]"
OTHER_TOKEN = "[OTHER_CITATION]"

# Internal sentinels are deliberately NOT valid citation tokens.
# They prevent citation-masking regexes from accidentally rematching
# [TARGET_CITATION] / [OTHER_CITATION] while processing the text.
_TARGET = "__TCIT_TARGET_8F91A__"
_OTHER = "__TCIT_OTHER_8F91A__"

FOLLOW_UP_NAMES = {
    0: "background_related_work",
    1: "comparison",
    2: "using_the_work",
    3: "extending_the_work",
}

In [ ]:
# ============================================================
# 3. DEPENDENCIES
# ============================================================

def ensure_import(package: str, import_name: Optional[str] = None):
    """
    Import a dependency; install it if missing.
    Intended for Colab / notebook execution.
    """
    import_name = import_name or package

    try:
        return __import__(import_name)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )
        return __import__(import_name)


_FOCAL_SEGMENTER = None


def get_focal_segmenter():
    """Create the pySBD segmenter only when FOCAL is actually loaded."""
    global _FOCAL_SEGMENTER

    if _FOCAL_SEGMENTER is None:
        pysbd = ensure_import("pysbd")
        _FOCAL_SEGMENTER = pysbd.Segmenter(
            language="en",
            clean=False,
            char_span=True,
        )

    return _FOCAL_SEGMENTER

In [ ]:
# ============================================================
# 4. GENERAL TEXT / CITATION UTILITIES
# ============================================================

YEAR_PATTERN = r"(?:18|19|20)\d{2}[a-z]?"
YEAR_RE = re.compile(rf"\b{YEAR_PATTERN}\b")

MULTICITE_SPAN_RE = re.compile(
    r"<span\b[^>]*>(.*?)</span>",
    flags=re.IGNORECASE | re.DOTALL,
)

SQUARE_GROUP_RE = re.compile(r"\[([^\[\]\n]{1,220})\]")
PAREN_GROUP_RE = re.compile(r"\(([^()\n]{1,400})\)")

NON_AUTHOR_WORDS = {
    "Fig", "Figure", "Figures",
    "Table", "Tables",
    "Eq", "Equation", "Equations",
    "Section", "Sections",
    "Appendix", "Appendices",
    "Chapter", "Chapters",
    "Page", "Pages",
    "Vol", "Volume",
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
}

# Narrative author-year forms such as:
#   Smith (2020)
#   Smith et al. (2020)
#   Smith and Jones (2020)
#   Smith & Jones (2020)
NARRATIVE_PAREN_RE = re.compile(
    rf"\b(?:"
    rf"[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]{{1,}}(?:\s+et\s+al\.?)?"
    rf"|[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]{{1,}}\s+(?:and|&)\s+"
    rf"[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]{{1,}}"
    rf")\s*\(\s*{YEAR_PATTERN}\s*\)"
)

# Bare author-year forms common in some extracted corpora:
#   Smith 2020
#   Smith et al. 2020
#   Smith & Jones 2020
BARE_AUTHOR_YEAR_RE = re.compile(
    rf"\b(?:"
    rf"[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]{{2,}}\s+et\s+al\.?"
    rf"|[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]{{2,}}\s+(?:and|&)\s+"
    rf"[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]{{2,}}"
    rf"|[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]{{2,}}"
    rf")\s*,?\s*{YEAR_PATTERN}\b"
)


def clean_whitespace(text: str) -> str:
    """HTML-decode and normalize whitespace only."""
    text = html.unescape(str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def strip_multicite_span_tags(text: str) -> str:
    """Remove MultiCite span tags while preserving their inner text."""
    text = re.sub(
        r"<span\b[^>]*>",
        "",
        str(text),
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"</span>",
        "",
        text,
        flags=re.IGNORECASE,
    )
    return text


def _looks_like_author_year_fragment(text: str) -> bool:
    """
    Conservative heuristic for parenthetical author-year citation fragments.
    Example: 'Smith et al., 2020' or 'Jones & Brown 2018'.
    """
    if not YEAR_RE.search(text):
        return False

    for year_match in YEAR_RE.finditer(text):
        before = text[max(0, year_match.start() - 90):year_match.start()]
        caps = re.findall(
            r"\b[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]{1,}\b",
            before,
        )
        caps = [word for word in caps if word not in NON_AUTHOR_WORDS]
        if caps:
            return True

    return False


def _numeric_reference_count(fragment: str) -> int:
    """
    Count references represented by a numeric fragment.
    Expands short numeric ranges: 12-14 -> 3.
    """
    fragment = fragment.strip()
    if not fragment:
        return 0

    range_match = re.fullmatch(
        r"(\d+)\s*[-–—]\s*(\d+)",
        fragment,
    )
    if range_match:
        start, end = map(int, range_match.groups())
        if 0 <= end - start <= 20:
            return end - start + 1
        return 2

    return len(re.findall(r"\d+", fragment))


def _render_numeric_group(content: str) -> str:
    """
    Convert a numeric citation cluster into atomic target/other tokens.

    Examples:
        "12, 13"               -> OTHER, OTHER
        TARGET, "13"           -> TARGET, OTHER
        OTHER, TARGET, "15"    -> OTHER, TARGET, OTHER
    """
    out: List[str] = []

    for chunk in re.split(r"[,;]", content):
        chunk = chunk.strip()
        if not chunk:
            continue

        # Preserve already-created internal sentinels.
        target_count = chunk.count(_TARGET)
        other_count = chunk.count(_OTHER)

        if target_count:
            out.extend([_TARGET] * target_count)
        if other_count:
            out.extend([_OTHER] * other_count)

        residual = chunk.replace(_TARGET, "").replace(_OTHER, "")
        residual = residual.strip()

        n_numeric = _numeric_reference_count(residual)
        if n_numeric:
            out.extend([_OTHER] * n_numeric)

        if (
            not target_count
            and not other_count
            and not n_numeric
            and residual
        ):
            out.append(residual)

    return ", ".join(out)


def _render_square_author_year_group(content: str) -> str:
    """
    Convert square-bracket author-year groups such as:
        [Smith et al., 2020]
        [Smith et al., 2020; Jones, 2021]
    while preserving an already inserted target sentinel.
    """
    parts = [part.strip() for part in content.split(";")]
    out: List[str] = []

    for part in parts:
        if not part:
            continue

        if _TARGET in part:
            out.append(_TARGET)

            residual = part.replace(_TARGET, "").replace(_OTHER, "")
            # Any separate year-bearing author fragment remaining in the same
            # part is treated as another citation.
            if _looks_like_author_year_fragment(residual):
                out.append(_OTHER)

            out.extend([_OTHER] * part.count(_OTHER))

        elif _OTHER in part:
            out.extend([_OTHER] * part.count(_OTHER))

            residual = part.replace(_OTHER, "")
            if _looks_like_author_year_fragment(residual):
                out.append(_OTHER)

        elif _looks_like_author_year_fragment(part):
            # Usually one cited work per semicolon-delimited part.
            out.append(_OTHER)

        else:
            out.append(part)

    return ", ".join(out)


def _mask_square_numeric_groups(text: str) -> str:
    """
    Replace numeric citation brackets with atomic [OTHER_CITATION] sentinels,
    while respecting a target sentinel already inserted by a source loader.
    """
    def repl(match: re.Match) -> str:
        content = match.group(1)
        stripped = content.strip()

        if _TARGET in content or _OTHER in content:
            # If the group also contains author-year text, use the
            # author-year-aware renderer; otherwise treat it as numeric.
            residual = content.replace(_TARGET, "").replace(_OTHER, "")
            if _looks_like_author_year_fragment(residual):
                return _render_square_author_year_group(content)
            return _render_numeric_group(content)

        if _looks_like_author_year_fragment(content):
            return _render_square_author_year_group(content)

        numeric_cluster = re.fullmatch(
            r"\d+[a-z]?"
            r"(?:\s*[-–—]\s*\d+[a-z]?)?"
            r"(?:\s*[,;]\s*"
            r"\d+[a-z]?"
            r"(?:\s*[-–—]\s*\d+[a-z]?)?"
            r")*",
            stripped,
            flags=re.IGNORECASE,
        )

        if numeric_cluster:
            return _render_numeric_group(content)

        return match.group(0)

    # Repeated passes also clean up some nested/malformed numeric extraction.
    for _ in range(4):
        new_text = SQUARE_GROUP_RE.sub(repl, text)
        if new_text == text:
            break
        text = new_text

    return text


def _parenthetical_group_replacer(match: re.Match) -> str:
    """
    Convert parenthetical author-year citation groups.

    Examples:
        (Smith, 2020) -> OTHER
        (Smith, 2020; Jones, 2021) -> (OTHER; OTHER)

    A target sentinel is never guessed. If a target is already present,
    it is preserved and only clearly separate author-year fragments are masked.
    """
    content = match.group(1)
    parts = [part.strip() for part in content.split(";")]

    # Simple standalone target wrapper:
    #   (__TARGET__) -> __TARGET__
    #   (e.g., __TARGET__) -> e.g., __TARGET__
    if len(parts) == 1 and _TARGET in content:
        without_target = content.replace(_TARGET, "").strip(" ,.;:")
        cue_removed = re.sub(
            r"^(?:e\.g\.|eg\.|see|cf\.|compare)\s*,?\s*",
            "",
            without_target,
            flags=re.IGNORECASE,
        ).strip()

        if not cue_removed:
            cue = without_target.strip()
            if cue:
                return f"{cue} {_TARGET}"
            return _TARGET

        # Complex mixed text: preserve parentheses here. Other citation
        # patterns will still be processed in later passes.
        return match.group(0)

    flags = [
        (_TARGET in part)
        or (_OTHER in part)
        or _looks_like_author_year_fragment(part)
        for part in parts
    ]

    if not any(flags):
        return match.group(0)

    rendered: List[str] = []
    replaced_any = False

    for part in parts:
        if _TARGET in part:
            rest = part.replace(_TARGET, "").strip(" ,.:")
            cue_removed = re.sub(
                r"^(?:e\.g\.|eg\.|see|cf\.|compare)\s*,?\s*",
                "",
                rest,
                flags=re.IGNORECASE,
            ).strip()

            if not cue_removed:
                rendered.append(_TARGET)
            else:
                rendered.append(part)
            replaced_any = True

        elif _OTHER in part:
            rendered.append(part)
            replaced_any = True

        elif _looks_like_author_year_fragment(part):
            rendered.append(_OTHER)
            replaced_any = True

        else:
            rendered.append(part)

    if not replaced_any:
        return match.group(0)

    if len(rendered) == 1 and rendered[0] in {_TARGET, _OTHER}:
        return rendered[0]

    return "(" + "; ".join(rendered) + ")"


def _mask_parenthetical_author_year_groups(text: str) -> str:
    return PAREN_GROUP_RE.sub(_parenthetical_group_replacer, text)


def _mask_narrative_author_year(text: str) -> str:
    """
    Mask narrative and bare author-year citation forms.
    """
    text = NARRATIVE_PAREN_RE.sub(_OTHER, text)

    def bare_replacer(match: re.Match) -> str:
        first_word = match.group(0).split()[0].strip(",")
        if first_word in NON_AUTHOR_WORDS:
            return match.group(0)
        return _OTHER

    text = BARE_AUTHOR_YEAR_RE.sub(bare_replacer, text)
    return text


def finalize_target_aware_text(text: str) -> str:
    """
    Source-independent finalization after a target sentinel has been inserted.

    The function:
      1) strips residual MultiCite span tags,
      2) masks other numeric citations,
      3) masks author-year citation groups,
      4) masks narrative/bare author-year citations,
      5) removes redundant wrappers around a lone target/other sentinel,
      6) restores [TARGET_CITATION] / [OTHER_CITATION].
    """
    text = html.unescape(str(text))
    text = strip_multicite_span_tags(text)

    text = _mask_square_numeric_groups(text)
    text = _mask_parenthetical_author_year_groups(text)
    text = _mask_narrative_author_year(text)

    # A second parenthetical pass catches groups whose internal citations
    # became sentinels after narrative masking.
    text = _mask_parenthetical_author_year_groups(text)

    # Standalone wrappers are redundant by project convention.
    text = re.sub(
        r"\(\s*" + re.escape(_TARGET) + r"\s*\)",
        _TARGET,
        text,
    )
    text = re.sub(
        r"\(\s*" + re.escape(_OTHER) + r"\s*\)",
        _OTHER,
        text,
    )
    text = re.sub(
        r"\[\s*" + re.escape(_TARGET) + r"\s*\]",
        _TARGET,
        text,
    )
    text = re.sub(
        r"\[\s*" + re.escape(_OTHER) + r"\s*\]",
        _OTHER,
        text,
    )

    # Remove malformed/redundant raw square brackets left adjacent to
    # already-resolved citation sentinels. This is safe because the sentinel
    # itself already denotes the complete citation marker.
    for sentinel in (_TARGET, _OTHER):
        escaped = re.escape(sentinel)
        text = re.sub(r"\[\s*(?=" + escaped + r")", "", text)
        text = re.sub(escaped + r"\s*\]", sentinel, text)

    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([,.;:])", r"\1", text)

    text = text.replace(_TARGET, TARGET_TOKEN)
    text = text.replace(_OTHER, OTHER_TOKEN)

    return text


def validate_target_aware_context(text: str) -> Tuple[bool, str]:
    """
    Every retained row must have exactly one TARGET token.
    """
    target_count = text.count(TARGET_TOKEN)

    if target_count != 1:
        return False, f"target_token_count_{target_count}"

    if "#CITATION_TAG" in text:
        return False, "raw_act2_target_marker_remaining"

    if re.search(r"<span\b|</span>", text, flags=re.IGNORECASE):
        return False, "raw_multicite_span_remaining"

    if "[[TARGET_CITATION]]" in text:
        return False, "double_square_target_wrapper"

    if "([TARGET_CITATION])" in text:
        return False, "redundant_parenthetical_target_wrapper"

    return True, "ok"


def possible_unmasked_citation(text: str) -> bool:
    """
    Diagnostic only. This does NOT remove rows.
    It flags contexts that may still contain an author-year/numeric citation
    that the conservative masker did not normalize.
    """
    if re.search(
        rf"\b[A-Z][A-Za-z'’\-]{{2,}}(?:\s+et\s+al\.?)?"
        rf"\s*,?\s*{YEAR_PATTERN}\b",
        text,
    ):
        return True

    if re.search(r"\[\s*\d+", text):
        return True

    return False


def add_audit(
    audit_rows: List[dict],
    source_dataset: str,
    source_id: str,
    reason: str,
    details: str = "",
) -> None:
    audit_rows.append({
        "source_dataset": source_dataset,
        "source_id": source_id,
        "reason": reason,
        "details": details,
    })

In [ ]:
# ============================================================
# 5. ACT2
# ============================================================

def _act2_target_instances(context: str) -> List[str]:
    """
    One target-aware instance per #CITATION_TAG occurrence.
    All sibling placeholders become [OTHER_CITATION].
    """
    matches = list(re.finditer(re.escape("#CITATION_TAG"), context))
    if not matches:
        return []

    instances: List[str] = []

    for target_index in range(len(matches)):
        pieces: List[str] = []
        last = 0

        for occurrence_index, match in enumerate(matches):
            pieces.append(context[last:match.start()])
            pieces.append(
                _TARGET
                if occurrence_index == target_index
                else _OTHER
            )
            last = match.end()

        pieces.append(context[last:])

        transformed = finalize_target_aware_text("".join(pieces))
        instances.append(transformed)

    return instances


def load_act2(
    path: Path,
    audit_rows: List[dict],
) -> Tuple[List[dict], Counter]:
    """
    ACT2 citation_class_label:
      0 BACKGROUND           -> follow_up 0
      1 COMPARES_CONTRASTS   -> follow_up 1
      2 EXTENSION            -> follow_up 3
      3 FUTURE               -> excluded
      4 MOTIVATION           -> excluded
      5 USES                 -> follow_up 2
    """
    mapping = {
        "0": (0, "BACKGROUND"),
        "1": (1, "COMPARES_CONTRASTS"),
        "5": (2, "USES"),
        "2": (3, "EXTENSION"),
    }

    rows: List[dict] = []
    stats = Counter()

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")

        required = {
            "unique_id",
            "citation_context",
            "citation_class_label",
        }
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(
                f"ACT2: missing required columns: {sorted(missing)}"
            )

        for record in reader:
            source_id = str(record.get("unique_id", "")).strip()
            original_code = str(
                record.get("citation_class_label", "")
            ).strip()

            if original_code not in mapping:
                reason = (
                    f"excluded_original_label_"
                    f"{original_code or 'EMPTY'}"
                )
                stats[reason] += 1
                add_audit(
                    audit_rows,
                    "ACT2",
                    source_id,
                    reason,
                )
                continue

            unified_label, original_name = mapping[original_code]
            context = str(record.get("citation_context", "") or "")

            if not context.strip():
                stats["empty_context"] += 1
                add_audit(
                    audit_rows,
                    "ACT2",
                    source_id,
                    "empty_context",
                )
                continue

            instances = _act2_target_instances(context)

            if not instances:
                stats["target_marker_not_found"] += 1
                add_audit(
                    audit_rows,
                    "ACT2",
                    source_id,
                    "target_marker_not_found",
                )
                continue

            if len(instances) > 1:
                stats["multi_target_contexts_expanded"] += 1
                stats["extra_target_instances_created"] += (
                    len(instances) - 1
                )

            for instance_index, target_context in enumerate(
                instances,
                start=1,
            ):
                valid, reason = validate_target_aware_context(
                    target_context
                )
                if not valid:
                    stats[reason] += 1
                    add_audit(
                        audit_rows,
                        "ACT2",
                        f"{source_id}::instance_{instance_index}",
                        reason,
                        target_context,
                    )
                    continue

                rows.append({
                    "source_dataset": "ACT2",
                    "citation": target_context,
                    "original_label": original_name,
                    "follow_up": unified_label,
                })
                stats["retained"] += 1

    return rows, stats

In [ ]:
# ============================================================
# 6. MULTICITE
# ============================================================

def _multicite_target_span_score(
    inner_text: str,
    sentence_text: str,
    span_start: int,
    span_end: int,
) -> int:
    """
    Prefer formal bibliographic targets over coreferential highlighted mentions.
    """
    inner = clean_whitespace(
        re.sub(r"<[^>]+>", "", inner_text)
    )

    score = 0

    if YEAR_RE.search(inner):
        score += 4

    if re.search(r"\[\s*\d|\d\s*\]", inner):
        score += 4

    left = sentence_text[max(0, span_start - 2):span_start]
    right = sentence_text[
        span_end:min(len(sentence_text), span_end + 2)
    ]

    if "[" in left or "]" in right:
        score += 3

    if re.search(r"\bet\s+al\.?\b", inner, flags=re.IGNORECASE):
        score += 2

    if re.search(
        r"\b[A-Z][A-Za-zÀ-ÖØ-öø-ÿ'’\-]{2,}\b",
        inner,
    ):
        score += 1

    coreference_only = {
        "they",
        "their",
        "them",
        "theirs",
        "that",
        "this method",
        "this approach",
        "this work",
        "this dataset",
        "their model",
        "their code",
        "this model",
        "this system",
    }

    if inner.lower() in coreference_only:
        score -= 3

    return score


def _choose_multicite_target_span(
    best_context_ids: Sequence[str],
    cite_ids: Sequence[str],
    sentence_map_raw: Dict[str, str],
) -> Optional[Tuple[str, int, str, int]]:
    """
    Return:
        (sentence_id, span_index_in_sentence, inner_span_text, score)

    Search order:
      1) annotated citation-bearing sentences in context order;
      2) any remaining annotated citation-bearing sentence;
      3) fallback: another sentence inside selected gold context.

    Within a tier, a formal bibliographic span is preferred.
    """
    ordered_cite_ids: List[str] = []
    seen = set()

    for sid in best_context_ids:
        if sid in cite_ids and sid not in seen:
            ordered_cite_ids.append(sid)
            seen.add(sid)

    for sid in sorted(cite_ids):
        if sid not in seen:
            ordered_cite_ids.append(sid)
            seen.add(sid)

    fallback_ids = [
        sid
        for sid in best_context_ids
        if sid not in seen
    ]

    for tier_ids in (ordered_cite_ids, fallback_ids):
        candidates = []

        for sid_order, sid in enumerate(tier_ids):
            sentence = sentence_map_raw.get(sid, "")

            for span_index, match in enumerate(
                MULTICITE_SPAN_RE.finditer(sentence)
            ):
                inner = match.group(1)
                score = _multicite_target_span_score(
                    inner,
                    sentence,
                    match.start(),
                    match.end(),
                )

                candidates.append({
                    "score": score,
                    "sid_order": sid_order,
                    "start": match.start(),
                    "sid": sid,
                    "span_index": span_index,
                    "inner": inner,
                })

        if candidates:
            candidates.sort(
                key=lambda item: (
                    -item["score"],
                    item["sid_order"],
                    item["start"],
                )
            )
            best = candidates[0]
            return (
                best["sid"],
                best["span_index"],
                best["inner"],
                best["score"],
            )

    return None


def _multicite_target_replacement(inner_text: str) -> str:
    """
    Preserve only numeric cluster punctuation that is structurally required.

    Examples:
      "8"                    -> TARGET
      "[11,"                 -> [TARGET,
      "13]"                  -> TARGET]
      "[5]"                  -> [TARGET]
      "Melamud et al. [11]" -> TARGET
    """
    inner = html.unescape(inner_text)
    stripped = inner.strip()

    numeric_fragment = re.fullmatch(
        r"[\[\]\d\s,;:\-–—]+",
        stripped,
    )

    if not numeric_fragment:
        return _TARGET

    prefix = "[" if stripped.startswith("[") else ""
    suffix = "]" if stripped.endswith("]") else ""

    trailing = ""
    no_close = stripped[:-1] if suffix else stripped
    no_close = no_close.rstrip()

    if no_close.endswith(","):
        trailing = ","
    elif no_close.endswith(";"):
        trailing = ";"

    return f"{prefix}{_TARGET}{trailing}{suffix}"


def _render_multicite_sentence(
    raw_sentence: str,
    current_sid: str,
    selected_sid: str,
    selected_span_index: int,
) -> str:
    """
    Replace exactly one selected yellow span with target sentinel.
    All other yellow spans lose only their HTML tags; their text remains.
    """
    current_index = 0

    def repl(match: re.Match) -> str:
        nonlocal current_index

        span_index = current_index
        current_index += 1
        inner = match.group(1)

        if (
            current_sid == selected_sid
            and span_index == selected_span_index
        ):
            return _multicite_target_replacement(inner)

        return inner

    return MULTICITE_SPAN_RE.sub(repl, raw_sentence)


def load_multicite(
    path: Path,
    audit_rows: List[dict],
) -> Tuple[List[dict], Counter]:
    """
    MultiCite mapping:
      @BACK@ -> follow_up 0
      @SIM@  -> follow_up 1
      @DIF@  -> follow_up 1
      @USE@  -> follow_up 2
      @EXT@  -> follow_up 3

    Excluded:
      @MOT@, @FUT@, @UNSURE@ and any other unmapped label.
    """
    mapping = {
        "@BACK@": (0, "BACKGROUND"),
        "@SIM@":  (1, "SIMILARITIES"),
        "@DIF@":  (1, "DIFFERENCES"),
        "@USE@":  (2, "USES"),
        "@EXT@":  (3, "EXTENDS"),
    }

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows: List[dict] = []
    stats = Counter()

    for item_id, obj in data.items():
        sentence_map_raw = {
            sentence.get("sent_id"): str(
                sentence.get("text", "") or ""
            )
            for sentence in obj.get("x", [])
            if sentence.get("sent_id")
        }

        occurrences = {}

        for raw_label, annotation in obj.get("y", {}).items():
            cite_sentence_ids = set(
                annotation.get("cite_sentences", []) or []
            )

            for gold_context in (
                annotation.get("gold_contexts", []) or []
            ):
                gold_context = tuple(gold_context)

                cite_ids = tuple(
                    sorted(
                        sid
                        for sid in gold_context
                        if sid in cite_sentence_ids
                    )
                )

                key = (
                    ("cite", cite_ids)
                    if cite_ids
                    else ("context", gold_context)
                )

                record = occurrences.setdefault(
                    key,
                    {
                        "labels": set(),
                        "contexts": [],
                        "cite_ids": set(),
                    },
                )

                record["labels"].add(raw_label)
                record["contexts"].append(gold_context)
                record["cite_ids"].update(cite_ids)

        for occurrence_index, occurrence in enumerate(
            occurrences.values(),
            start=1,
        ):
            source_id = f"{item_id}::occurrence_{occurrence_index}"
            raw_labels = occurrence["labels"]

            mapped_classes = {
                mapping[label][0]
                for label in raw_labels
                if label in mapping
            }

            non_target_labels = {
                label
                for label in raw_labels
                if label not in mapping
            }

            if not mapped_classes:
                stats["only_non_target_labels"] += 1
                add_audit(
                    audit_rows,
                    "MultiCite",
                    source_id,
                    "only_non_target_labels",
                    "|".join(sorted(raw_labels)),
                )
                continue

            if non_target_labels:
                stats["target_plus_non_target_labels"] += 1
                add_audit(
                    audit_rows,
                    "MultiCite",
                    source_id,
                    "target_plus_non_target_labels",
                    "|".join(sorted(raw_labels)),
                )
                continue

            if len(mapped_classes) != 1:
                stats["multiple_unified_target_classes"] += 1
                add_audit(
                    audit_rows,
                    "MultiCite",
                    source_id,
                    "multiple_unified_target_classes",
                    "|".join(sorted(raw_labels)),
                )
                continue

            unified_label = next(iter(mapped_classes))

            best_context_ids = max(
                occurrence["contexts"],
                key=lambda ids: (
                    len(ids),
                    sum(
                        len(sentence_map_raw.get(sid, ""))
                        for sid in ids
                    ),
                ),
            )

            selected = _choose_multicite_target_span(
                best_context_ids,
                occurrence["cite_ids"],
                sentence_map_raw,
            )

            if selected is None:
                stats["target_span_not_found"] += 1
                add_audit(
                    audit_rows,
                    "MultiCite",
                    source_id,
                    "target_span_not_found",
                    "|".join(best_context_ids),
                )
                continue

            (
                selected_sid,
                selected_span_index,
                selected_inner,
                selected_score,
            ) = selected

            if selected_score >= 3:
                stats["target_resolution_formal"] += 1
            else:
                stats["target_resolution_mention_fallback"] += 1

            context_parts = []

            for sid in best_context_ids:
                if sid not in sentence_map_raw:
                    continue

                context_parts.append(
                    _render_multicite_sentence(
                        sentence_map_raw[sid],
                        current_sid=sid,
                        selected_sid=selected_sid,
                        selected_span_index=selected_span_index,
                    )
                )

            context = finalize_target_aware_text(
                " ".join(context_parts)
            )

            if not context:
                stats["empty_context"] += 1
                add_audit(
                    audit_rows,
                    "MultiCite",
                    source_id,
                    "empty_context",
                )
                continue

            valid, reason = validate_target_aware_context(context)
            if not valid:
                stats[reason] += 1
                add_audit(
                    audit_rows,
                    "MultiCite",
                    source_id,
                    reason,
                    context,
                )
                continue

            original_names = sorted({
                mapping[label][1]
                for label in raw_labels
                if label in mapping
            })

            rows.append({
                "source_dataset": "MultiCite",
                "citation": context,
                "original_label": "|".join(original_names),
                "follow_up": unified_label,
            })
            stats["retained"] += 1

    return rows, stats

In [ ]:
# ============================================================
# 7. FOCAL
# ============================================================

def normalize_focal_label(label: str) -> str:
    label = str(label).strip().lower()
    return re.sub(r"\s+", " ", label)


def _resolve_focal_span_end(
    paragraph: str,
    start: int,
    end: int,
    expected_text: str = "",
) -> Tuple[int, bool]:
    """
    Convert a FOCAL end offset to Python-exclusive form.

    The dataset generally behaves as start:end. For robustness, if the
    corresponding Citation Text is available, also test start:end+1.

    Returns:
        (exclusive_end, exact_text_match_used)
    """
    expected = str(expected_text or "")

    if expected:
        direct = paragraph[start:end]
        plus_one = (
            paragraph[start:end + 1]
            if end < len(paragraph)
            else ""
        )

        if direct == expected:
            return end, True

        if plus_one == expected:
            return end + 1, True

        # Whitespace-normalized fallback.
        if clean_whitespace(direct) == clean_whitespace(expected):
            return end, True

        if (
            plus_one
            and clean_whitespace(plus_one)
            == clean_whitespace(expected)
        ):
            return end + 1, True

    return end, False


def _focal_spans_can_collapse(
    sentence_text: str,
    local_spans: Sequence[Tuple[int, int]],
) -> bool:
    """
    True when gaps between split target pieces contain only citation glue.
    This allows several pieces of one FOCAL citation to be collapsed into
    one target token without deleting normal sentence content.
    """
    if len(local_spans) <= 1:
        return True

    citation_glue_re = re.compile(
        r"^[\s,;:/&()\[\]\-–—]*"
        r"(?:(?:and|&|et\s+al\.?)"
        r"[\s,;:/&()\[\]\-–—]*)?$",
        flags=re.IGNORECASE,
    )

    spans = sorted(local_spans)

    for (_, end_a), (start_b, _) in zip(spans, spans[1:]):
        gap = sentence_text[end_a:start_b]
        if not citation_glue_re.fullmatch(gap):
            return False

    return True


def _insert_focal_target(
    sentence_text: str,
    local_spans: Sequence[Tuple[int, int]],
) -> str:
    """
    Insert exactly one target sentinel using FOCAL's annotated target spans.
    """
    spans = sorted(local_spans)

    if not spans:
        return ""

    if _focal_spans_can_collapse(sentence_text, spans):
        start = spans[0][0]
        end = spans[-1][1]
        return (
            sentence_text[:start]
            + _TARGET
            + sentence_text[end:]
        )

    # Conservative fallback: replace the first target piece with TARGET
    # and delete later pieces, preserving all intervening normal text.
    replacements = []

    for index, (start, end) in enumerate(spans):
        replacements.append(
            (start, end, _TARGET if index == 0 else "")
        )

    out = sentence_text

    for start, end, replacement in reversed(replacements):
        out = out[:start] + replacement + out[end:]

    return out


def extract_focal_target_aware_sentence(
    record: dict,
) -> Tuple[str, str]:
    """
    Return:
        (target_aware_sentence, status)

    Uses exact FOCAL "Citation Start End" offsets in the original Paragraph.
    """
    paragraph = str(record.get("Paragraph", "") or "")
    raw_spans = record.get("Citation Start End", []) or []
    citation_texts = record.get("Citation Text", []) or []

    if not paragraph:
        return "", "empty_paragraph"

    if not raw_spans:
        return "", "missing_citation_spans"

    spans: List[Tuple[int, int]] = []
    exact_match_count = 0

    try:
        for index, span in enumerate(raw_spans):
            start = int(span[0])
            end = int(span[1])

            if start < 0 or end <= start or start >= len(paragraph):
                return "", "invalid_citation_span"

            expected = (
                citation_texts[index]
                if index < len(citation_texts)
                else ""
            )

            exclusive_end, matched = _resolve_focal_span_end(
                paragraph,
                start,
                end,
                expected,
            )

            if (
                exclusive_end <= start
                or exclusive_end > len(paragraph)
            ):
                return "", "invalid_citation_span"

            if matched:
                exact_match_count += 1

            spans.append((start, exclusive_end))

    except (TypeError, ValueError, IndexError):
        return "", "invalid_citation_span"

    sentences = get_focal_segmenter().segment(paragraph)
    matched_sentence_indices = set()

    for citation_start, citation_end in spans:
        found = False

        for sentence_index, sentence in enumerate(sentences):
            sent_start = sentence.start
            sent_end = sentence.end

            if (
                sent_start < citation_end
                and citation_start < sent_end
            ):
                matched_sentence_indices.add(sentence_index)
                found = True
                break

        if not found:
            return "", "citation_span_not_in_sentence"

    if len(matched_sentence_indices) != 1:
        return "", "target_spans_cross_sentences"

    sentence_index = next(iter(matched_sentence_indices))
    sentence_obj = sentences[sentence_index]

    sent_start = sentence_obj.start
    sent_end = sentence_obj.end
    sentence_text = sentence_obj.sent

    local_spans = []

    for start, end in spans:
        local_start = max(start, sent_start) - sent_start
        local_end = min(end, sent_end) - sent_start

        if local_end <= local_start:
            return "", "invalid_local_target_span"

        local_spans.append((local_start, local_end))

    with_target = _insert_focal_target(
        sentence_text,
        local_spans,
    )

    context = finalize_target_aware_text(with_target)

    valid, reason = validate_target_aware_context(context)
    if not valid:
        return "", reason

    if citation_texts and exact_match_count < len(spans):
        return context, "resolved_offset_text_mismatch_fallback"

    return context, "resolved"


def load_focal(
    audit_rows: List[dict],
) -> Tuple[List[dict], Counter]:
    """
    Download/load FOCAL from Hugging Face and build target-aware rows.

    Mapping:
      Background                       -> follow_up 0
      Compare/Contrast                 -> follow_up 1
      Similarities                     -> follow_up 1
      Differences                      -> follow_up 1
      Uses                             -> follow_up 2
      Extends                          -> follow_up 3

    Excluded:
      Motivation / Motivations
      Future Work
      any other unmapped label
    """
    datasets_module = ensure_import("datasets")
    load_dataset = datasets_module.load_dataset

    print("\nDownloading/loading FOCAL from Hugging Face...")
    dataset = load_dataset("adsabs/FOCAL")

    focal_mapping = {
        "background": (0, "BACKGROUND"),

        "compare/contrast": (1, "COMPARE_CONTRAST"),
        "compare contrast": (1, "COMPARE_CONTRAST"),
        "compare and contrast": (1, "COMPARE_CONTRAST"),
        "similarities": (1, "SIMILARITIES"),
        "similarity": (1, "SIMILARITIES"),
        "differences": (1, "DIFFERENCES"),
        "difference": (1, "DIFFERENCES"),
        "diferences": (1, "DIFFERENCES"),

        "uses": (2, "USES"),
        "use": (2, "USES"),

        "extends": (3, "EXTENDS"),
        "extension": (3, "EXTENDS"),
        "extending": (3, "EXTENDS"),
    }

    rows: List[dict] = []
    stats = Counter()

    for split_name, split in dataset.items():
        for row_number, record in enumerate(split, start=1):
            identifier = str(
                record.get("Identifier", "")
                or f"{split_name}_{row_number}"
            )
            source_id = f"{split_name}::{identifier}"

            raw_labels = record.get("Functions Label", []) or []

            normalized_to_original = defaultdict(set)

            for raw in raw_labels:
                if raw is None:
                    continue

                raw_text = str(raw).strip()
                if not raw_text:
                    continue

                normalized_to_original[
                    normalize_focal_label(raw_text)
                ].add(raw_text)

            normalized_labels = set(normalized_to_original)

            mapped_classes = {
                focal_mapping[label][0]
                for label in normalized_labels
                if label in focal_mapping
            }

            non_target_labels = {
                label
                for label in normalized_labels
                if label not in focal_mapping
            }

            if not mapped_classes:
                stats["only_non_target_labels"] += 1
                add_audit(
                    audit_rows,
                    "FOCAL",
                    source_id,
                    "only_non_target_labels",
                    "|".join(sorted(normalized_labels)),
                )
                continue

            if non_target_labels:
                stats["target_plus_non_target_labels"] += 1
                add_audit(
                    audit_rows,
                    "FOCAL",
                    source_id,
                    "target_plus_non_target_labels",
                    "|".join(sorted(normalized_labels)),
                )
                continue

            if len(mapped_classes) != 1:
                stats["multiple_unified_target_classes"] += 1
                add_audit(
                    audit_rows,
                    "FOCAL",
                    source_id,
                    "multiple_unified_target_classes",
                    "|".join(sorted(normalized_labels)),
                )
                continue

            unified_label = next(iter(mapped_classes))

            context, status = (
                extract_focal_target_aware_sentence(record)
            )

            if not context:
                stats[status] += 1
                add_audit(
                    audit_rows,
                    "FOCAL",
                    source_id,
                    status,
                )
                continue

            if status == "resolved":
                stats["target_resolution_exact"] += 1
            else:
                stats[status] += 1

            context_stripped = context.strip()

            # Preserve the old fragment filtering policy.
            if context_stripped.startswith(
                ("=", ",", ";", ":", ")", "]")
            ):
                stats["sentence_fragment"] += 1
                add_audit(
                    audit_rows,
                    "FOCAL",
                    source_id,
                    "sentence_fragment",
                    context,
                )
                continue

            if len(context_stripped.split()) < 5:
                stats["sentence_fragment"] += 1
                add_audit(
                    audit_rows,
                    "FOCAL",
                    source_id,
                    "sentence_fragment",
                    context,
                )
                continue

            original_labels = []

            for normalized_label in sorted(normalized_labels):
                if normalized_label in focal_mapping:
                    original_labels.append(
                        focal_mapping[normalized_label][1]
                    )
                else:
                    original_labels.append(
                        "|".join(
                            sorted(
                                normalized_to_original[
                                    normalized_label
                                ]
                            )
                        )
                    )

            rows.append({
                "source_dataset": "FOCAL",
                "citation": context,
                "original_label": "|".join(
                    sorted(set(original_labels))
                ),
                "follow_up": unified_label,
            })
            stats["retained"] += 1

    return rows, stats

In [ ]:
# ============================================================
# 8. ALREADY TARGET-AWARE CSV SOURCES
# ============================================================

def normalize_existing_target_aware_context(text: str) -> str:
    """
    Normalize only redundant wrappers around existing target-aware tokens.

    Examples:
        ([TARGET_CITATION]) -> [TARGET_CITATION]
        ([OTHER_CITATION])  -> [OTHER_CITATION]
        [[TARGET_CITATION]] -> [TARGET_CITATION]

    Parentheses around true multi-citation clusters such as
    ([TARGET_CITATION]; [OTHER_CITATION]) are preserved.
    """
    text = html.unescape(str(text or ""))
    text = re.sub(r"\s+", " ", text).strip()

    previous = None
    while text != previous:
        previous = text
        text = re.sub(
            r"\(\s*" + re.escape(TARGET_TOKEN) + r"\s*\)",
            TARGET_TOKEN,
            text,
        )
        text = re.sub(
            r"\(\s*" + re.escape(OTHER_TOKEN) + r"\s*\)",
            OTHER_TOKEN,
            text,
        )
        text = re.sub(
            r"\[\s*" + re.escape(TARGET_TOKEN) + r"\s*\]",
            TARGET_TOKEN,
            text,
        )
        text = re.sub(
            r"\[\s*" + re.escape(OTHER_TOKEN) + r"\s*\]",
            OTHER_TOKEN,
            text,
        )

    return re.sub(r"\s+", " ", text).strip()


def load_required_target_aware_csv(
    path: Path,
    source_name: str,
) -> Tuple[List[dict], Counter]:
    """
    Load an already target-aware citation-function CSV.

    Required input columns:
        citation
        follow_up

    The final unified source_dataset is set to source_name:
        JOURNAL_REAL or SYNTHETIC.

    No rows are silently dropped. Any malformed row stops execution.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"{source_name} file not found:\n{path}"
        )

    rows: List[dict] = []
    stats = Counter()

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)

        required = {"citation", "follow_up"}
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(
                f"{source_name}: missing required columns: "
                f"{sorted(missing)}"
            )

        for row_number, raw in enumerate(reader, start=2):
            stats["input_rows"] += 1

            context = normalize_existing_target_aware_context(
                raw.get("citation", "")
            )
            if not context:
                raise ValueError(
                    f"{source_name}: empty citation at CSV row {row_number}."
                )

            try:
                follow_up = int(str(raw.get("follow_up", "")).strip())
            except (TypeError, ValueError):
                raise ValueError(
                    f"{source_name}: invalid follow_up at CSV row "
                    f"{row_number}: {raw.get('follow_up')!r}"
                )

            if follow_up not in FOLLOW_UP_NAMES:
                raise ValueError(
                    f"{source_name}: follow_up={follow_up} at CSV row "
                    f"{row_number}; expected 0, 1, 2, or 3."
                )

            valid, reason = validate_target_aware_context(context)
            if not valid:
                raise ValueError(
                    f"{source_name}: target-aware validation failed at "
                    f"CSV row {row_number}: {reason}\n{context}"
                )

            rows.append({
                "source_dataset": source_name,
                "citation": context,
                "original_label": FOLLOW_UP_NAMES[follow_up],
                "follow_up": follow_up,
            })
            stats["retained"] += 1

    if stats["retained"] != stats["input_rows"]:
        raise RuntimeError(
            f"{source_name}: not all rows were retained: "
            f"{stats['retained']}/{stats['input_rows']}."
        )

    return rows, stats

In [ ]:
# ============================================================
# 9. OUTPUT + VALIDATION
# ============================================================

def write_csv(rows: Sequence[dict], output_path: Path) -> None:
    fieldnames = [
        "source_dataset",
        "citation",
        "original_label",
        "follow_up",
    ]

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(
        output_path,
        "w",
        encoding="utf-8-sig",
        newline="",
    ) as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def write_audit_csv(
    audit_rows: Sequence[dict],
    output_path: Path,
) -> None:
    fieldnames = [
        "source_dataset",
        "source_id",
        "reason",
        "details",
    ]

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(
        output_path,
        "w",
        encoding="utf-8-sig",
        newline="",
    ) as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(audit_rows)


def validate_final_dataset(rows: Sequence[dict]) -> None:
    if not rows:
        raise ValueError("Final dataset is empty.")

    for row_index, row in enumerate(rows, start=1):
        context = str(row.get("citation", ""))

        valid, reason = validate_target_aware_context(context)

        if not valid:
            raise ValueError(
                f"Final row {row_index} failed target-aware "
                f"validation: {reason}\n{context}"
            )

        try:
            label = int(row.get("follow_up"))
        except (TypeError, ValueError):
            raise ValueError(
                f"Final row {row_index} has invalid follow_up."
            )

        if label not in FOLLOW_UP_NAMES:
            raise ValueError(
                f"Final row {row_index} has follow_up={label}."
            )


def print_report(
    all_rows: Sequence[dict],
    source_stats: Dict[str, Counter],
    audit_rows: Sequence[dict],
) -> None:
    print("\n" + "=" * 78)
    print("FINAL TARGET-AWARE UNIFIED DATASET")
    print("=" * 78)

    print(f"\nTotal retained rows: {len(all_rows):,}")

    by_source = Counter(
        row["source_dataset"]
        for row in all_rows
    )

    print("\nRows by source dataset:")
    for source in ["ACT2", "MultiCite", "FOCAL", "JOURNAL_REAL", "SYNTHETIC"]:
        print(f"  {source:10s}: {by_source[source]:,}")

    by_follow_up = Counter(
        int(row["follow_up"])
        for row in all_rows
    )

    print("\nRows by follow_up:")
    for label in range(4):
        print(
            f"  {label} = "
            f"{FOLLOW_UP_NAMES[label]:24s}: "
            f"{by_follow_up[label]:,}"
        )

    target_counts = Counter(
        row["citation"].count(TARGET_TOKEN)
        for row in all_rows
    )

    print("\n[TARGET_CITATION] count per retained row:")
    for count, n_rows in sorted(target_counts.items()):
        print(f"  {count}: {n_rows:,}")

    with_other = sum(
        OTHER_TOKEN in row["citation"]
        for row in all_rows
    )

    print(
        f"\nRows containing at least one "
        f"[OTHER_CITATION]: {with_other:,}"
    )

    possible_leftovers = sum(
        possible_unmasked_citation(row["citation"])
        for row in all_rows
    )

    print(
        "Rows flagged by conservative diagnostic as possibly "
        f"containing an unmasked citation: {possible_leftovers:,}"
    )
    print(
        "  NOTE: diagnostic only; these rows are NOT removed."
    )

    print("\nSource-specific processing statistics:")
    for source, counter in source_stats.items():
        print(f"\n  {source}:")
        if not counter:
            print("    none")
        else:
            for reason, count in counter.most_common():
                print(f"    {reason}: {count:,}")

    print(f"\nAudit rows written: {len(audit_rows):,}")

    print("\nMapping used:")
    print("  0 = background_related_work")
    print("  1 = comparison")
    print("  2 = using_the_work")
    print("  3 = extending_the_work")

    print("\nTarget-aware tokens:")
    print(f"  {TARGET_TOKEN} = labeled target citation")
    print(f"  {OTHER_TOKEN} = every other detectable citation")

    print("=" * 78)

In [ ]:
# ============================================================
# 10. MAIN
# ============================================================

def main() -> None:
    if not ACT2_PATH.exists():
        raise FileNotFoundError(
            f"ACT2 file not found:\n{ACT2_PATH}"
        )

    if not MULTICITE_PATH.exists():
        raise FileNotFoundError(
            f"MultiCite file not found:\n{MULTICITE_PATH}"
        )

    if not JOURNAL_PATH.exists():
        raise FileNotFoundError(
            f"JOURNAL_REAL file not found:\n{JOURNAL_PATH}"
        )

    if not SYNTHETIC_PATH.exists():
        raise FileNotFoundError(
            f"SYNTHETIC file not found:\n{SYNTHETIC_PATH}"
        )

    audit_rows: List[dict] = []

    print("Loading ACT2...")
    act2_rows, act2_stats = load_act2(
        ACT2_PATH,
        audit_rows,
    )
    print(f"  retained ACT2 rows: {len(act2_rows):,}")

    print("\nLoading MultiCite...")
    multicite_rows, multicite_stats = load_multicite(
        MULTICITE_PATH,
        audit_rows,
    )
    print(
        f"  retained MultiCite rows: "
        f"{len(multicite_rows):,}"
    )

    focal_rows, focal_stats = load_focal(audit_rows)
    print(f"  retained FOCAL rows: {len(focal_rows):,}")

    print("\nLoading JOURNAL_REAL...")
    journal_rows, journal_stats = load_required_target_aware_csv(
        JOURNAL_PATH,
        "JOURNAL_REAL",
    )
    print(f"  retained JOURNAL_REAL rows: {len(journal_rows):,}")

    print("\nLoading SYNTHETIC...")
    synthetic_rows, synthetic_stats = load_required_target_aware_csv(
        SYNTHETIC_PATH,
        "SYNTHETIC",
    )
    print(f"  retained SYNTHETIC rows: {len(synthetic_rows):,}")

    all_rows = (
        act2_rows
        + multicite_rows
        + focal_rows
        + journal_rows
        + synthetic_rows
    )

    final_source_counts = Counter(
        row["source_dataset"] for row in all_rows
    )

    if final_source_counts["JOURNAL_REAL"] != len(journal_rows):
        raise RuntimeError(
            "JOURNAL_REAL was not fully included in the final dataset."
        )

    if final_source_counts["SYNTHETIC"] != len(synthetic_rows):
        raise RuntimeError(
            "SYNTHETIC was not fully included in the final dataset."
        )

    validate_final_dataset(all_rows)

    write_csv(all_rows, OUTPUT_CSV)
    write_audit_csv(audit_rows, AUDIT_CSV)

    print_report(
        all_rows,
        {
            "ACT2": act2_stats,
            "MultiCite": multicite_stats,
            "FOCAL": focal_stats,
            "JOURNAL_REAL": journal_stats,
            "SYNTHETIC": synthetic_stats,
        },
        audit_rows,
    )

    print(f"\nCSV saved to:\n{OUTPUT_CSV}")
    print(f"\nAudit CSV saved to:\n{AUDIT_CSV}")


if __name__ == "__main__":
    main()